In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Circuit 타이밍 시각화

<Accordion>
<AccordionItem title="패키지 버전">

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Qiskit에 내장된 [타임라인 드로어](/guides/visualize-circuit-timing)는 정적 Circuit에 유용하지만, 브로드캐스팅 및 분기 결정과 같은 암묵적 작업으로 인해 [동적 Circuit](/guides/classical-feedforward-and-control-flow)의 타이밍을 정확하게 반영하지 못할 수 있습니다. 동적 Circuit 지원의 일환으로, Qiskit Runtime은 요청 시 Job 결과 내에 정확한 Circuit 타이밍 정보를 반환합니다.

> **Note:** - 이것은 실험적 기능입니다. 미리 보기 출시 상태에 있으며 변경될 수 있습니다.
> - 이 기능은 Qiskit Runtime Sampler Job에만 적용됩니다.
> - 총 Circuit 시간이 "compilation" 메타데이터에 반환되지만, 이것은 청구에 사용되는 시간(QPU 시간)이 아닙니다.
### 타이밍 데이터 검색 활성화
타이밍 데이터 검색을 활성화하려면 Primitive Job을 실행할 때 실험적 `scheduler_timing` 플래그를 `True`로 설정하세요.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Circuit 타이밍 데이터 접근
요청 시, 각 PUB에 대한 Circuit 타이밍 데이터가 `["compilation"]["scheduler_timing"]["timing"]` 아래의 Job 결과 메타데이터에 반환됩니다. 이 필드에는 원시 타이밍 정보가 포함됩니다. 타이밍 정보를 표시하려면 [타이밍 시각화](#visualize-timings) 섹션에 설명된 내장 시각화 도구를 사용하세요.

다음 코드를 사용하여 첫 번째 PUB의 Circuit 타이밍 데이터에 접근하세요:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### 원시 타이밍 데이터 이해
`draw_circuit_schedule_timing` 메서드를 사용하여 Circuit 타이밍 데이터를 시각화하는 것이 가장 일반적인 사용 사례이지만, 반환된 원시 타이밍 데이터의 구조를 이해하는 것이 유용할 수 있습니다. 예를 들어 정보를 프로그래밍 방식으로 추출하는 데 도움이 될 수 있습니다.

`["compilation"]["scheduler_timing"]["timing"]`에서 반환된 타이밍 데이터는 문자열 목록입니다. 각 문자열은 일부 채널에서의 단일 명령을 나타내며 다음 데이터 유형으로 쉼표로 구분됩니다:

- `Branch` - 명령이 제어 흐름(then / else) 또는 메인 브랜치에 있는지 여부를 결정합니다.
- `Instruction` - 게이트와 작동할 Qubit.
- `Channel` - 명령이 할당되는 채널. 다음 중 하나일 수 있습니다:
      - `Qubit x` - Qubit _x_의 드라이브 채널.
      - `AWGRx_y` (임의 파형 생성기 읽기) - Qubit을 측정할 때 통신하기 위해 읽기 채널에서 사용됩니다. _x_ 및 _y_ 인수는 각각 읽기 기기 ID와 Qubit 번호에 해당합니다.
- `T0` - 완전한 스케줄 내에서 명령 시작 시간
- `Duration` - _dt_ 초 단위의 명령 지속 시간, 여기서 1 dt = 1 스케줄링 사이클. [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt)를 사용하여 백엔드의 `dt` 값을 찾을 수 있습니다.
- `Pulse` - 사용 중인 펄스 작업 유형.

예제:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### 타이밍 시각화
`qiskit-ibm-runtime` v0.43.0 이상에서는 Circuit 타이밍을 시각화할 수 있습니다. 타이밍을 시각화하려면 먼저 [`draw_circuit_schedule_timing` 메서드](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26)를 사용하여 결과 메타데이터를 `fig`로 변환해야 합니다. 이 메서드는 직접 표시하거나 파일에 저장하거나 두 가지 모두 가능한 `plotly` 그림을 반환합니다. 사용할 `plotly` 명령에 대한 자세한 내용은 [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) 및 [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html)을 참조하세요.

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![위에 마우스를 올리면 시작, 종료, 지속 시간 등의 정보가 표시됩니다.](../docs/images/guides/visualize-circuit-timing/image_1.svg '생성된 그림의 예')
#### 생성된 그림 이해
`draw_circuit_schedule_timing`이 출력하는 Circuit 타이밍 데이터 이미지는 다음 정보를 전달합니다:

- X 축은 _dt_ 초 단위의 시간이며, 여기서 1 dt = 1 스케줄링 사이클. [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt)를 사용하여 백엔드의 `dt` 값을 찾을 수 있습니다.
- Y 축은 채널입니다(채널을 펄스를 방출하는 기기로 생각하세요).
    - `Receive channel` - 기기 자체가 아닌 유일한 채널. 해당 시간에 허브와의 통신 절차에 참여하는 모든 채널에서 재생되는 명령입니다.
    - `Qubit x` - Qubit x의 드라이브 채널.
    - `AWGRx_y` (임의 파형 생성기 읽기) - Qubit을 측정할 때 통신하기 위해 읽기 채널에서 사용됩니다. _x_ 및 _y_ 인수는 각각 읽기 기기 ID와 Qubit 번호에 해당합니다.
    - `Hub` - 브로드캐스팅을 제어합니다.

또한 각 명령은 *X_Y* 형식을 가지며, *X*는 명령의 이름이고 *Y*는 펄스 유형입니다. `play` 유형은 제어 펄스를 적용하고, `capture`는 Qubit의 상태를 기록합니다. 각 명령 위에 마우스를 올려 더 자세한 내용을 볼 수도 있습니다. 예를 들어, 이전 그림은 1161 dt에서 Qubit 10에 적용된 X 게이트에 대한 제어 펄스를 보여줍니다.
### 전체 예제
이 예제는 옵션을 활성화하고, 메타데이터에서 Circuit 타이밍 정보를 가져와 이미지로 표시하는 방법을 보여줍니다.

먼저 환경을 설정하고, Circuit을 정의하여 ISA Circuit으로 변환하고, Job을 정의하고 실행합니다.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

다음으로 Circuit 스케줄 타이밍을 가져옵니다: